# 04 · Does It Help a Portfolio? Backtesting Regime-Based Allocation
**Week 4 of _Can a Neural Network See Market Regimes Coming?_**

We now use regimes to decide **how to split money** between three assets each day:

| Regime | Equity (Nifty) | Gold | Cash |
|---|---|---|---|
| Calm | 80% | 10% | 10% |
| Turbulent | 40% | 30% | 30% |
| Crisis | 10% | 50% | 40% |

and compare four strategies on the **test period (2019 onwards)**:
1. **Buy & hold Nifty**
2. **60/40:** always 60% Nifty, 40% cash
3. **HMM (filtered):** today's regime estimated by the HMM using **only past data**
4. **Neural network:** the regime predicted for today by the network from notebook 03

**Golden rule of backtesting:** the weights used on day *t* must be decided with information available at the
end of day *t − 1*. Breaking this rule (lookahead bias) is the most common mistake in trading research.

In [ ]:
!pip -q install yfinance hmmlearn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from hmmlearn.hmm import GaussianHMM
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

np.random.seed(0)
torch.manual_seed(0)
plt.rcParams["figure.figsize"] = (11, 4)

## 1. Rebuild the pipeline from notebooks 01–03 (same code, fewer printouts)

In [ ]:
def load_close(ticker, start="2008-01-01"):
    """Download daily closing prices from Yahoo Finance as a pandas Series."""
    df = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    close = df["Close"]
    if isinstance(close, pd.DataFrame):   # newer yfinance returns a 1-column DataFrame
        close = close.iloc[:, 0]
    return close.dropna()

def make_features(close, vol_window=20):
    """Daily log return + rolling annualised volatility."""
    df = pd.DataFrame({"close": close})
    df["ret"] = np.log(df["close"]).diff()                                   # log return
    df["vol"] = df["ret"].rolling(vol_window).std() * np.sqrt(252)           # annualised volatility
    return df.dropna()

REGIME_NAMES = ["Calm", "Turbulent", "Crisis"]
SPLIT_DATE = "2019-01-01"        # everything before = training data, from here on = test data

def fit_hmm(df, n_states=3, seed=42):
    """Fit an HMM on TRAINING data only. Returns (model, mean, std, order)."""
    X = df[["ret", "vol"]].values
    train = df.index < SPLIT_DATE
    mean, std = X[train].mean(axis=0), X[train].std(axis=0)
    Xs = (X - mean) / std
    model = GaussianHMM(n_components=n_states, covariance_type="full",
                        n_iter=500, random_state=seed)
    model.fit(Xs[train])
    order = np.argsort(model.means_[:, 1])        # sort states by volatility: low -> high
    return model, mean, std, order

def hmm_labels(df, model, mean, std, order):
    """Most likely regime for every day (Viterbi), renamed so 0=Calm, 1=Turbulent, 2=Crisis."""
    Xs = (df[["ret", "vol"]].values - mean) / std
    raw = model.predict(Xs)
    rename = {old: new for new, old in enumerate(order)}
    return np.array([rename[s] for s in raw])

LOOKBACK = 20      # the network sees the last 20 days of features

def make_windows(df, mean, std):
    """X[i] = last LOOKBACK days of standardised [ret, vol] up to day t, y[i] = regime on day t+1."""
    feats = (df[["ret", "vol"]].values - mean) / std
    regime = df["regime"].values
    X, y, today, dates = [], [], [], []
    for t in range(LOOKBACK - 1, len(df) - 1):
        X.append(feats[t - LOOKBACK + 1 : t + 1].flatten())
        y.append(regime[t + 1])       # target: TOMORROW's regime
        today.append(regime[t])       # today's regime (used for the simple baseline)
        dates.append(df.index[t + 1])
    return (np.array(X, dtype=np.float32), np.array(y), np.array(today), pd.DatetimeIndex(dates))

class RegimeMLP(nn.Module):
    def __init__(self, n_inputs, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_inputs, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, n_classes),          # raw scores ("logits") for each regime
        )
    def forward(self, x):
        return self.net(x)

def train_model(X_train, y_train, X_val, y_val, epochs=60, lr=1e-3, verbose=True):
    model = RegimeMLP(X_train.shape[1])
    # Crisis days are rare, so we weight each class by 1 / (how often it appears).
    counts = np.bincount(y_train, minlength=3)
    weights = torch.tensor(len(y_train) / (3 * np.maximum(counts, 1)), dtype=torch.float32)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
                        batch_size=64, shuffle=True)
    Xv, yv = torch.tensor(X_val), torch.tensor(y_val)
    history, best_val, best_state = [], float("inf"), None

    for epoch in range(epochs):
        model.train()
        total = 0.0
        for xb, yb in loader:
            opt.zero_grad()                     # 1. clear old gradients
            loss = loss_fn(model(xb), yb)       # 2. forward pass + loss
            loss.backward()                     # 3. backpropagation
            opt.step()                          # 4. update the weights
            total += loss.item() * len(xb)
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xv), yv).item()
        history.append((total / len(X_train), val_loss))
        if val_loss < best_val:                 # keep the best model on validation data
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if verbose and (epoch + 1) % 10 == 0:
            print(f"epoch {epoch+1:3d}  train loss {history[-1][0]:.3f}  val loss {val_loss:.3f}")

    model.load_state_dict(best_state)
    return model, history

nifty = load_close("^NSEI")
df = make_features(nifty)
hmm, mean, std, order = fit_hmm(df)
df["regime"] = hmm_labels(df, hmm, mean, std, order)

X, y, y_today, dates = make_windows(df, mean, std)
is_train = dates < "2016-01-01"
is_val = (dates >= "2016-01-01") & (dates < SPLIT_DATE)
is_test = dates >= SPLIT_DATE
nn_model, _ = train_model(X[is_train], y[is_train], X[is_val], y[is_val], verbose=False)

nn_model.eval()
with torch.no_grad():
    nn_pred = nn_model(torch.tensor(X[is_test])).argmax(dim=1).numpy()
# nn_pred is indexed by the day it is FOR, and was computed from data up to the day before. No lookahead.
nn_signal = pd.Series(nn_pred, index=dates[is_test], name="nn")
print("Network ready. Test days:", len(nn_signal))

## 2. HMM regimes using only past data ("filtered")
`predict()` uses the whole sequence (it peeks ahead). Instead, for each day we run the HMM on data **up to that day**
and take the most likely state for the **last** day. That is what you could actually have known at the time.
This takes a few seconds.

In [ ]:
Xs = (df[["ret", "vol"]].values - mean) / std
rename = {old: new for new, old in enumerate(order)}
test_pos = np.where(df.index >= SPLIT_DATE)[0]

filtered = []
for t in test_pos:
    probs = hmm.predict_proba(Xs[: t + 1])[-1]     # P(state on day t | data up to day t)
    filtered.append(rename[int(np.argmax(probs))])
# Regime known at the END of day t is used for day t+1, hence the shift(1).
hmm_signal = pd.Series(filtered, index=df.index[test_pos]).shift(1).dropna().astype(int)

## 3. Asset returns
- **Equity:** Nifty 50 (`^NSEI`)
- **Gold:** Nippon India Gold ETF (`GOLDBEES.NS`), gold priced in rupees
- **Cash:** assumed to earn a fixed **6.5% per year** (roughly Indian short-term rates). This is a simplification.

In [ ]:
gold = load_close("GOLDBEES.NS")
assets = pd.DataFrame({"equity": nifty.pct_change(), "gold": gold.pct_change()})
assets = assets[assets.index >= SPLIT_DATE].dropna(subset=["equity"])
assets["gold"] = assets["gold"].fillna(0.0)            # gold ETF missing a day -> assume no change
assets["cash"] = (1.065) ** (1 / 252) - 1
assets.describe()

## 4. Run the strategies

In [ ]:
ALLOC = {  # regime -> (equity, gold, cash)
    0: (0.80, 0.10, 0.10),   # Calm
    1: (0.40, 0.30, 0.30),   # Turbulent
    2: (0.10, 0.50, 0.40),   # Crisis
}

def regime_strategy(signal):
    """Daily portfolio return when each day's weights come from that day's regime signal."""
    sig = signal.reindex(assets.index).ffill().dropna().astype(int)
    w = np.array([ALLOC[s] for s in sig])
    return pd.Series((assets.loc[sig.index, ["equity", "gold", "cash"]].values * w).sum(axis=1), index=sig.index)

returns = pd.DataFrame({
    "Buy & hold Nifty": assets["equity"],
    "60/40": 0.6 * assets["equity"] + 0.4 * assets["cash"],
    "HMM (filtered)": regime_strategy(hmm_signal),
    "Neural network": regime_strategy(nn_signal),
}).dropna()

## 5. Performance metrics
- **CAGR:** average yearly growth · **Volatility:** yearly std of returns
- **Sharpe ratio:** extra return over cash per unit of volatility (higher = better reward for the risk taken)
- **Sortino ratio:** like Sharpe, but only counts *downside* volatility
- **Max drawdown:** worst fall from a previous peak · **Calmar:** CAGR / |max drawdown|

In [ ]:
def metrics(r, cash_daily=assets["cash"].iloc[0]):
    growth = (1 + r).cumprod()
    years = len(r) / 252
    cagr = growth.iloc[-1] ** (1 / years) - 1
    vol = r.std() * np.sqrt(252)
    excess = r - cash_daily
    sharpe = excess.mean() / excess.std() * np.sqrt(252)
    downside = excess[excess < 0].std() * np.sqrt(252)
    sortino = excess.mean() * 252 / downside
    max_dd = (growth / growth.cummax() - 1).min()
    return pd.Series({"CAGR": cagr, "Volatility": vol, "Sharpe": sharpe,
                      "Sortino": sortino, "Max drawdown": max_dd, "Calmar": cagr / abs(max_dd)})

table = returns.apply(metrics).T
table.style.format({"CAGR": "{:.1%}", "Volatility": "{:.1%}", "Sharpe": "{:.2f}",
                    "Sortino": "{:.2f}", "Max drawdown": "{:.1%}", "Calmar": "{:.2f}"})

In [ ]:
growth = (1 + returns).cumprod()
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
growth.plot(ax=ax[0], title="Growth of ₹1 (test period)")
(growth / growth.cummax() - 1).plot(ax=ax[1], title="Drawdown", legend=False)
plt.tight_layout(); plt.show()

## 6. Be honest about the results
This backtest ignores things that matter in real life:
- **Transaction costs:** switching regimes means trading; every switch costs money. Count how often each strategy switches.
- **One test period:** 2019 onwards contains one big crash (March 2020). Results could differ in other periods.
- **The allocation table is a guess.** Changing it changes the results; tuning it on the test period would be cheating.
- **Cash at a fixed 6.5%** is a simplification.

A strategy that only "wins" because of one lucky period, or that ignores costs, is not a real win.

## Check your understanding
1. Why do we `shift(1)` the HMM signal? What would go wrong without it?
2. What is the difference between the Viterbi labels (`predict`) and the filtered labels used here?
3. Which matters more for an investor, a higher CAGR or a smaller max drawdown? It depends on what?
4. How would you add transaction costs to `regime_strategy`?
5. Did the neural network add anything over the HMM alone? How would you test that more convincingly?